# Joint Optimization of Ranking and Calibration (JRC All)


In [ ]:
import time
from datetime import datetime
import gc
import random
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score, log_loss, ndcg_score
from collections import defaultdict

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Dataset
    TORCH_AVAILABLE = True
    TORCH_IMPORT_ERROR = None
except ImportError as e:
    torch = nn = F = DataLoader = Dataset = None
    TORCH_AVAILABLE = False
    TORCH_IMPORT_ERROR = e


def ensure_torch_available():
    if not TORCH_AVAILABLE:
        raise ImportError(
            "chapter_5_projects_all/jrc-ranking_all.ipynb 需要 PyTorch 环境。"
            "请先安装 torch，再重新运行本 notebook。"
        ) from TORCH_IMPORT_ERROR


# ============================================================================
# 🎯 核心配置
# ============================================================================
MODE = "offline"  # 👈 与 4.recall_all, 5.feature_engineering_all 保持一致
SEED = 2020

np.random.seed(SEED)
random.seed(SEED)
if TORCH_AVAILABLE:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

print("=" * 70)
print(f"🔧 JRC 排序模型 - 当前运行模式: {MODE}")
if MODE == "offline":
    print("🔬 离线验证模式: 使用验证集评估模型效果")
else:
    print("🚀 在线提交模式: 使用全量数据训练，生成提交结果")
print(f"🧠 PyTorch 可用: {'是' if TORCH_AVAILABLE else '否'}")
print("=" * 70)

offline = (MODE == "offline")


## 1. 数据加载

加载由 `5.feature_engineering.ipynb` 生成的特征数据，包括：
- 训练集 / 验证集 / 测试集特征表（每行为一个 (用户, 候选文章) 对）
- 用户历史点击行为序列（DIN 注意力机制所需）

In [ ]:
# 数据路径（独立项目版：纯相对路径）
project_root = Path.cwd().resolve()
data_path = project_root / 'data' / 'raw' / 'news_recommendation'
save_path = project_root / 'data' / 'processed' / 'temp_results'
output_path = project_root / 'outputs'

for path in [data_path, save_path, output_path]:
    path.mkdir(parents=True, exist_ok=True)

print(f"📊 offline = {offline} (基于 MODE = '{MODE}')")
print(f"📁 项目根目录: {project_root}")
print(f"📁 原始数据路径: {data_path}")
print(f"📁 中间结果路径: {save_path}")
print(f"📁 导出结果路径: {output_path}")


In [ ]:
# 加载特征数据
print("📂 正在加载特征数据...")
trn_user_item_feats_df = pd.read_csv(save_path / 'trn_user_item_feats_df_all.csv')
trn_user_item_feats_df['click_article_id'] = trn_user_item_feats_df['click_article_id'].astype(int)
print(f"   训练集特征: {trn_user_item_feats_df.shape}")

if offline:
    val_user_item_feats_df = pd.read_csv(save_path / 'val_user_item_feats_df_all.csv')
    val_user_item_feats_df['click_article_id'] = val_user_item_feats_df['click_article_id'].astype(int)
    print(f"   验证集特征: {val_user_item_feats_df.shape}")
else:
    val_user_item_feats_df = None
    print("   ⏭️ 跳过验证集加载 (在线模式)")

tst_user_item_feats_df = pd.read_csv(save_path / 'tst_user_item_feats_df_all.csv')
if len(tst_user_item_feats_df) > 0:
    tst_user_item_feats_df['click_article_id'] = tst_user_item_feats_df['click_article_id'].astype(int)
print(f"   测试集特征: {tst_user_item_feats_df.shape}")

if 'label' in tst_user_item_feats_df.columns:
    del tst_user_item_feats_df['label']

if offline and len(tst_user_item_feats_df) == 0:
    print("   ℹ️  离线模式下测试集为空是正常的（只需验证集评估）")

print("✅ 特征数据加载完毕！")


你看到的输出：
每个括号里的两个数字，分别代表：

- 第一个数字：样本数量（行数），即有多少条 user-item 对
- 第二个数字：特征数量（列数），即每条样本有多少个特征
举例说明：

- `(275112, 28)`：表示训练集有 275112 条样本，每条样本有 28 个特征。用户历史点击每人 5-10 条，召回每人 150 条。200,000 用户 × 150 候选 = 30,000,000 条。#正样本：200,000（每人1个正样本）负样本：~29,800,000（其余都是负样本）。负采样！将负样本大幅压缩，最终：200,000 正样本 + ~75,000 负样本 = 275112 条。
- `(5582058, 28)`：表示验证集有 5582058 条样本，每条样本有 28 个特征。验证集构造，不做负采样！保留全部候选用于完整评估，最终：5,582,058 条（全部召回候选）。
- `(0, 29)`：表示测试集没有样本（行数为 0），但每条样本本应有 29 个特征
如果测试集是 `(0, 29)`，说明测试集特征文件没有数据，通常是召回或特征工程阶段没有生成测试集样本。

In [ ]:
# 看训练集前 3 条样本，转置后更容易观察每个字段
display(trn_user_item_feats_df.head(3).T)

# 看验证集
if val_user_item_feats_df is not None:
    display(val_user_item_feats_df.head(3).T)

# 看测试集
if len(tst_user_item_feats_df) > 0:
    display(tst_user_item_feats_df.head(3).T)
else:
    print("测试集为空")


In [ ]:
print(trn_user_item_feats_df.columns)

In [ ]:
# 加载用户历史点击行为（DIN Attention 所需）
print("📂 加载历史点击行为数据...")

if offline:
    all_data = pd.read_csv(save_path / 'click_hist_all.csv')
    print(f"   [离线模式] 使用 leave-last-out 历史: {all_data.shape}")
else:
    trn_data = pd.read_csv(data_path / 'train_click_log.csv')
    tst_data = pd.read_csv(data_path / 'testA_click_log.csv')
    all_data = pd.concat([trn_data, tst_data]).reset_index(drop=True)
    print(f"   [在线模式] 合并训练集+测试集: {all_data.shape}")

hist_click = all_data[['user_id', 'click_article_id']].groupby('user_id').agg({list}).reset_index()
his_behavior_df = pd.DataFrame()
his_behavior_df['user_id'] = hist_click['user_id']
his_behavior_df['hist_click_article_id'] = hist_click['click_article_id']

print(f"   用户行为序列数: {len(his_behavior_df)}")
print("✅ 行为数据加载完成！")


In [ ]:
print(his_behavior_df)

In [ ]:
his_behavior_df['seq_len'] = his_behavior_df['hist_click_article_id'].apply(len)

# 每个长度有多少用户
len_dist = his_behavior_df['seq_len'].value_counts().sort_index()

print("用户数:", len(his_behavior_df))
print("最短序列:", his_behavior_df['seq_len'].min())
print("最长序列:", his_behavior_df['seq_len'].max())
print("平均长度:", his_behavior_df['seq_len'].mean())
print("中位数:", his_behavior_df['seq_len'].median())

display(len_dist.head(20))


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
plt.bar(len_dist.index, len_dist.values, width=0.8)
plt.yscale("log")
plt.xlabel("History Sequence Length")
plt.ylabel("Number of Users (log scale)")
plt.title("User Click History Length Distribution (Log Y)")
plt.grid(axis='y', alpha=0.3)
plt.show()



In [ ]:
cdf = len_dist.cumsum() / len_dist.sum()

plt.figure(figsize=(12, 5))
plt.plot(cdf.index, cdf.values)
plt.xlabel("History Sequence Length")
plt.ylabel("Cumulative Ratio of Users")
plt.title("CDF of User History Length")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
print(all_data.head(3).T)

1.  **[离线模式] 使用训练集：(1112623, 9)**
    - 代表训练集有 1,112,623 条点击记录（行），每条有 9 个字段（如 `user_id`、`click_article_id`、时间等）。
    - 这里的每一行是“用户点击某篇文章”的一次行为。

2.  **用户行为序列数：200000**
    - 代表有 200,000 个不同的用户（`user_id`）。
    - 这是对训练集按 `user_id` 分组后，统计有多少个独立用户。

In [ ]:
display(all_data.head())
print(all_data.shape)
print(all_data.columns.tolist())
print(all_data.dtypes)
display(his_behavior_df.head())
print(his_behavior_df.shape)

print(type(his_behavior_df.iloc[0]['hist_click_article_id']))
print(his_behavior_df.iloc[0]['hist_click_article_id'])
print(len(his_behavior_df.iloc[0]['hist_click_article_id']))

uid = his_behavior_df.iloc[0]['user_id']
print("user_id =", uid)

print("\n这个用户在原始点击日志里的记录：")
display(all_data[all_data['user_id'] == uid])

print("\n这个用户聚合后的点击序列：")
print(his_behavior_df[his_behavior_df['user_id'] == uid]['hist_click_article_id'].iloc[0])



In [ ]:
# 合并行为序列到特征表
trn_df = trn_user_item_feats_df.merge(his_behavior_df, on='user_id')

if offline:
    val_df = val_user_item_feats_df.merge(his_behavior_df, on='user_id')
else:
    val_df = None

# 测试集：只在非空时合并
if len(tst_user_item_feats_df) > 0:
    tst_df = tst_user_item_feats_df.merge(his_behavior_df, on='user_id')
else:
    tst_df = tst_user_item_feats_df  # 保持为空 DataFrame

print(f"✅ 合并后训练集: {trn_df.shape}")
if val_df is not None:
    print(f"   合并后验证集: {val_df.shape}")
if len(tst_df) > 0:
    print(f"   合并后测试集: {tst_df.shape}")
else:
    print(f"   合并后测试集: 空（离线模式正常）")

## 2. 特征处理

将特征分为三类，每类在 JRC 模型中的处理方式不同：

| 特征类型 | 示例 | 模型处理 |
|----------|------|----------|
| **稀疏特征** | `user_id`, `click_article_id`, `category_id` | → Embedding → 拼接 |
| **稠密特征** | `sim0`, `time_diff0`, `click_size` | → MinMaxScaler 归一化 → 直接拼接 |
| **行为序列** | `hist_click_article_id` | → Embedding → DIN Attention → 加权池化 |

In [ ]:
# 定义特征分组
sparse_fea = [
    'user_id', 'click_article_id', 'category_id',
    'click_environment', 'click_deviceGroup', 'click_os',
    'click_country', 'click_region', 'click_referrer_type',
]

dense_fea = [
    'sim0', 'time_diff0', 'word_diff0',
    'sim_max', 'sim_min', 'sim_sum', 'sim_mean',
    'score', 'rank', 'click_size', 'time_diff_mean',
    'active_level', 'user_time_hob1', 'user_time_hob2',
    'words_hbo', 'words_count',
]

hist_behavior_fea = ['hist_click_article_id']

print(f"稀疏特征 ({len(sparse_fea)}): {sparse_fea}")
print(f"稠密特征 ({len(dense_fea)}): {dense_fea}")
print(f"行为序列 ({len(hist_behavior_fea)}): {hist_behavior_fea}")

In [ ]:
# Dense 特征归一化
print("📊 对 Dense 特征进行 MinMaxScaler 归一化...")
mm = MinMaxScaler()

for feat in dense_fea:
    trn_df[feat] = mm.fit_transform(trn_df[[feat]])
    if offline and val_df is not None:
        val_df[feat] = mm.transform(val_df[[feat]])
    # 保护：只有测试集非空时才归一化
    if len(tst_df) > 0:
        tst_df[feat] = mm.transform(tst_df[[feat]])
    else:
        print(f"   ⚠️ 测试集为空，跳过 {feat} 归一化")

print("✅ Dense 特征归一化完成！")
if len(tst_df) == 0:
    print("⚠️  警告：测试集为空，请检查 4.recall.ipynb 和 5.feature_engineering.ipynb 是否正确运行")

## 3. 构建 PyTorch 输入与模型


In [ ]:
emb_dim = 8
max_len = 50

# 只包含非空的 DataFrame
all_frames = [trn_df] + ([val_df] if val_df is not None else []) + ([tst_df] if len(tst_df) > 0 else [])

# ====== 稀疏特征 ID 映射 ======
sparse_maps = {}
for feat in sparse_fea:
    vals = pd.concat([f[feat].astype('int64') for f in all_frames], ignore_index=True)
    # click_article_id 还要包含历史序列中的 ID
    if feat == 'click_article_id':
        hist_vals = []
        for f in all_frames:
            for seq in f['hist_click_article_id']:
                if isinstance(seq, (list, np.ndarray)):
                    hist_vals.extend([int(x) for x in seq])
        if len(hist_vals) > 0:
            vals = pd.concat([vals, pd.Series(hist_vals, dtype='int64')], ignore_index=True)
    _, uniques = pd.factorize(vals, sort=False)
    mapping = {int(val): int(idx) for idx, val in enumerate(uniques)}
    if feat == 'click_article_id':
        mapping = {k: v + 1 for k, v in mapping.items()}  # 保留 0 用于 padding
    sparse_maps[feat] = mapping


def _vocab_size(mapping: dict) -> int:
    return (max(mapping.values()) + 1) if mapping else 1


sparse_vocab_sizes = {feat: _vocab_size(sparse_maps[feat]) for feat in sparse_fea}
click_vocab_size = _vocab_size(sparse_maps['click_article_id'])
dense_input_dim = len(dense_fea)

print("✅ 稀疏特征 ID 映射构建完成")
for feat in sparse_fea:
    print(f"   {feat}: vocab_size = {sparse_vocab_sizes[feat]}")
print(f"   hist_click_article_id 共享 click_article_id vocab_size = {click_vocab_size}")


In [ ]:
for feat in sparse_fea:
    print(f"{feat} 前10个映射: {list(sparse_maps[feat].items())[:10]}")


vocab_size 就是"这个特征有多少种不同的值"，模型要为每种值学习一个 Embedding 向量！

In [ ]:
# ====== PyTorch JRC 配置 ======
jrc_config = {
    'shared_dnn_units': [256, 128],
    'logit_head_units': [64],
    'gate_hidden_units': [32],
    'attention_hidden_units': [64, 32],
    'linear_logits': True,
    'dropout_rate': 0.1,
}

print("🏗️ PyTorch JRC 配置准备完成")
print(f"   稀疏特征数: {len(sparse_fea)}")
print(f"   稠密特征数: {dense_input_dim}")
print(f"   Embedding dim: {emb_dim}")
print(f"   Max history len: {max_len}")


group=['dnn']：这些特征会被送入 DNN（深度神经网络）模型进行学习和建模。

In [ ]:
# ====== 构建模型输入字典 ======
def pad_sequences_np(sequences, maxlen, value=0):
    padded = np.full((len(sequences), maxlen), value, dtype=np.int64)
    for idx, seq in enumerate(sequences):
        if len(seq) == 0:
            continue
        trunc = seq[:maxlen]
        padded[idx, :len(trunc)] = np.asarray(trunc, dtype=np.int64)
    return padded


def build_model_input(df: pd.DataFrame) -> dict:
    """将 DataFrame 转换为模型输入字典"""
    model_input = {}
    # 稀疏特征：映射为连续 ID
    for feat in sparse_fea:
        model_input[feat] = (
            df[feat].astype('int64').map(sparse_maps[feat]).fillna(0).astype('int64').values
        )
    # 稠密特征：直接使用归一化后的值
    for feat in dense_fea:
        model_input[feat] = df[feat].astype('float32').values
    # 变长稀疏特征：映射 + pad
    cmap = sparse_maps['click_article_id']
    raw_seq_list = df['hist_click_article_id'].tolist()
    mapped_seq = []
    for seq in raw_seq_list:
        if isinstance(seq, (list, np.ndarray)):
            mapped_seq.append([int(cmap.get(int(x), 0)) if int(x) != 0 else 0 for x in seq])
        else:
            mapped_seq.append([0])
    model_input['hist_click_article_id'] = pad_sequences_np(mapped_seq, max_len, value=0)
    return model_input


print("🔄 构建模型输入...")
x_trn = build_model_input(trn_df)
y_trn = trn_df['label'].values.astype('float32')

if offline and val_df is not None:
    x_val = build_model_input(val_df)
    y_val = val_df['label'].values.astype('float32')

# 测试集：只在非空时构建输入
if len(tst_df) > 0:
    x_tst = build_model_input(tst_df)
else:
    x_tst = None

print(f"   训练集样本数: {len(y_trn)}, 正样本率: {y_trn.mean():.4f}")
if offline:
    print(f"   验证集样本数: {len(y_val)}, 正样本率: {y_val.mean():.4f}")
if x_tst is not None:
    print(f"   测试集样本数: {len(tst_df)}")
else:
    print(f"   测试集: 空（离线模式跳过）")
print("✅ 模型输入构建完成！")


## 4. 构建 PyTorch JRC 模型


In [ ]:
ensure_torch_available()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def make_mlp(input_dim, hidden_units, dropout_rate=0.0, output_dim=None, output_activation=None):
    layers = []
    prev_dim = input_dim
    for units in hidden_units:
        layers.append(nn.Linear(prev_dim, units))
        layers.append(nn.ReLU())
        if dropout_rate > 0:
            layers.append(nn.Dropout(dropout_rate))
        prev_dim = units
    if output_dim is not None:
        layers.append(nn.Linear(prev_dim, output_dim))
        if output_activation == 'sigmoid':
            layers.append(nn.Sigmoid())
    return nn.Sequential(*layers), (output_dim if output_dim is not None else prev_dim)


class DINAttentionPooling(nn.Module):
    def __init__(self, emb_dim, hidden_units):
        super().__init__()
        self.att_mlp, _ = make_mlp(
            emb_dim * 4,
            hidden_units,
            dropout_rate=0.0,
            output_dim=1,
            output_activation=None,
        )

    def forward(self, query_emb, hist_emb, mask):
        query = query_emb.unsqueeze(1).expand_as(hist_emb)
        att_input = torch.cat([query, hist_emb, query - hist_emb, query * hist_emb], dim=-1)
        scores = self.att_mlp(att_input).squeeze(-1)

        all_pad = ~mask.any(dim=1)
        scores = scores.masked_fill(~mask, -1e9)
        scores = scores.masked_fill(all_pad.unsqueeze(1), 0.0)

        weights = torch.softmax(scores, dim=1)
        weights = weights * mask.float()
        denom = weights.sum(dim=1, keepdim=True).clamp_min(1e-8)
        weights = weights / denom

        pooled = torch.bmm(weights.unsqueeze(1), hist_emb).squeeze(1)
        pooled = pooled.masked_fill(all_pad.unsqueeze(1), 0.0)
        return pooled


class PyTorchJRC(nn.Module):
    def __init__(self, sparse_vocab_sizes, dense_input_dim, emb_dim, max_len, config):
        super().__init__()
        self.sparse_fea = list(sparse_vocab_sizes.keys())
        self.dense_fea = dense_fea
        self.emb_dim = emb_dim
        self.max_len = max_len
        self.use_linear = config.get('linear_logits', True)

        self.sparse_embeddings = nn.ModuleDict()
        for feat, vocab_size in sparse_vocab_sizes.items():
            if feat == 'click_article_id':
                self.sparse_embeddings[feat] = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
            else:
                self.sparse_embeddings[feat] = nn.Embedding(vocab_size, emb_dim)

        if self.use_linear:
            self.linear_sparse_embeddings = nn.ModuleDict({
                feat: nn.Embedding(vocab_size, 1, padding_idx=0 if feat == 'click_article_id' else None)
                for feat, vocab_size in sparse_vocab_sizes.items()
            })
            self.linear_dense = nn.Linear(dense_input_dim, 1)

        self.din_pooling = DINAttentionPooling(emb_dim, config.get('attention_hidden_units', [64, 32]))

        shared_input_dim = len(self.sparse_fea) * emb_dim + dense_input_dim + emb_dim
        self.shared_backbone, shared_output_dim = make_mlp(
            shared_input_dim,
            config.get('shared_dnn_units', [256, 128]),
            dropout_rate=config.get('dropout_rate', 0.0),
            output_dim=None,
        )

        self.logit_head, logit_hidden_dim = make_mlp(
            shared_output_dim,
            config.get('logit_head_units', [64]),
            dropout_rate=config.get('dropout_rate', 0.0),
            output_dim=None,
        )
        self.logits_out = nn.Linear(logit_hidden_dim, 2)

        self.gate_head, gate_hidden_dim = make_mlp(
            shared_output_dim,
            config.get('gate_hidden_units', [32]),
            dropout_rate=config.get('dropout_rate', 0.0),
            output_dim=None,
        )
        self.gate_out = nn.Sequential(nn.Linear(gate_hidden_dim, 1), nn.Sigmoid())

    def forward(self, batch):
        sparse_emb_list = [self.sparse_embeddings[feat](batch[feat]) for feat in self.sparse_fea]
        dense_tensor = torch.stack([batch[feat] for feat in dense_fea], dim=1)

        target_emb = self.sparse_embeddings['click_article_id'](batch['click_article_id'])
        hist_ids = batch['hist_click_article_id']
        hist_emb = self.sparse_embeddings['click_article_id'](hist_ids)
        hist_mask = hist_ids.ne(0)
        hist_repr = self.din_pooling(target_emb, hist_emb, hist_mask)

        dnn_input = torch.cat(sparse_emb_list + [dense_tensor, hist_repr], dim=1)
        shared_repr = self.shared_backbone(dnn_input)

        logits_hidden = self.logit_head(shared_repr)
        logits_2d = self.logits_out(logits_hidden)

        if self.use_linear:
            linear_sparse = sum(
                self.linear_sparse_embeddings[feat](batch[feat]).squeeze(-1)
                for feat in self.sparse_fea
            )
            linear_dense = self.linear_dense(dense_tensor).squeeze(-1)
            linear_logit = linear_sparse + linear_dense
            linear_2d = torch.stack([-linear_logit, linear_logit], dim=1)
            logits_2d = logits_2d + linear_2d

        gate_hidden = self.gate_head(shared_repr)
        gate_lambda = self.gate_out(gate_hidden).squeeze(-1)
        return logits_2d, gate_lambda



def count_parameters(model):
    return sum(param.numel() for param in model.parameters() if param.requires_grad)


def init_jrc_model():
    model = PyTorchJRC(sparse_vocab_sizes, dense_input_dim, emb_dim, max_len, jrc_config).to(DEVICE)
    return model


print("🏗️ 构建 PyTorch JRC 模型...")
model = init_jrc_model()
print(model)
print(f"📦 设备: {DEVICE}")
print(f"📊 可训练参数量: {count_parameters(model):,}")


## 5. 联合损失函数详解

JRC 的核心是基于 **2D logit** 的联合损失函数：

$$L_{total} = \lambda \cdot L_{ranking} + (1 - \lambda) \cdot L_{calibration}$$

其中：
- 模型输出 2D logit $\mathbf{z} = [z_0, z_1]$，排序分数 $s = z_1 - z_0$
- $L_{ranking}$：**Pairwise BPR Loss**，在 batch 内对排序分数 $s$ 做正负样本配对：
  $$L_{rank} = -\frac{1}{|P||N|}\sum_{i \in P}\sum_{j \in N} \log \sigma(s_i - s_j)$$
- $L_{calibration}$：**2-class Cross-Entropy**（on 2D logits, from_logits=True）：
  $$L_{calib} = -\frac{1}{n}\sum_i \log \text{softmax}(\mathbf{z}_i)[y_i]$$
  等价于 $-\frac{1}{n}\sum_i [y_i \log \sigma(s_i) + (1-y_i)\log(1-\sigma(s_i))]$
- $\lambda$：由 **Context Gate** 产生，对每个样本不同，取 batch 均值后加权


### Context Gate 的直觉

- 对于 **"容易区分"** 的样本（正负很分明），$\lambda \to 0$ 偏向校准 → 学精确概率
- 对于 **"难区分"** 的样本（排序边界附近），$\lambda \to 1$ 偏向排序 → 把顺序排对

## JRC 联合损失函数原理与公式详解

### 1. 排序损失（Pairwise BPR Loss）

- **目标**：让正样本的排序分数高于负样本。
- **数学公式**：
  $$
  L_{\text{BPR}} = -\frac{1}{|P||N|} \sum_{i \in P} \sum_{j \in N} \log \sigma(s_i - s_j)
  $$
  其中 $s_i$ 是正样本分数，$s_j$ 是负样本分数，$\sigma$ 是 sigmoid。
- **实现要点**：
  - 对 batch 内所有正负样本两两配对，计算分数差。
  - 用 sigmoid 拉开正负分数差距，log 取均值。

### 2. ListNet 排序损失（可选）

- **目标**：让预测分布和真实分布尽量一致。
- **数学公式**：
  $$
  L_{\text{ListNet}} = -\sum_{i} P(y_i) \log P(\hat{y}_i)
  $$
  其中 $P(y_i)$ 是标签 softmax，$P(\hat{y}_i)$ 是预测分数 softmax。

### 3. 联合排序+校准损失（JRC 核心）

- **2D logit 输出**：$\mathbf{z} = [z_0, z_1]$
  - 排序分数 $s = z_1 - z_0$
  - 校准概率 $p = \text{softmax}(\mathbf{z})[1] = \sigma(z_1 - z_0)$
- **损失函数**：
  $$
  L_{\text{total}} = \lambda \cdot L_{\text{BPR}} + (1-\lambda) \cdot L_{\text{calib}}
  $$
  其中 $\lambda$ 由 Context Gate 动态产生。
- **校准损失**：
  $$
  L_{\text{calib}} = -\frac{1}{N} \sum_{i} \log \text{softmax}(\mathbf{z}_i)[y_i]
  $$
  即 2-class cross-entropy（from_logits=True 自动 softmax）。




In [ ]:
ensure_torch_available()


class RankingDictDataset(Dataset):
    def __init__(self, x_dict, y=None):
        self.x_dict = x_dict
        self.y = y
        self.length = len(next(iter(x_dict.values())))

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        sample = {key: value[idx] for key, value in self.x_dict.items()}
        if self.y is None:
            return sample
        return sample, self.y[idx]



def collate_with_labels(batch):
    features, labels = zip(*batch)
    collated = {}
    for key in features[0]:
        arr = np.stack([item[key] for item in features])
        dtype = torch.long if key in sparse_fea or key == 'hist_click_article_id' else torch.float32
        collated[key] = torch.tensor(arr, dtype=dtype)
    return collated, torch.tensor(np.asarray(labels), dtype=torch.float32)



def collate_features(batch):
    collated = {}
    for key in batch[0]:
        arr = np.stack([item[key] for item in batch])
        dtype = torch.long if key in sparse_fea or key == 'hist_click_article_id' else torch.float32
        collated[key] = torch.tensor(arr, dtype=dtype)
    return collated



def make_loader(x_dict, y=None, batch_size=256, shuffle=False):
    dataset = RankingDictDataset(x_dict, y)
    if y is None:
        return DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_features)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_with_labels)



def move_batch_to_device(batch):
    return {key: value.to(DEVICE) for key, value in batch.items()}



def pairwise_bpr_loss(y_true, y_pred):
    pos_scores = y_pred[y_true == 1]
    neg_scores = y_pred[y_true == 0]
    if pos_scores.numel() == 0 or neg_scores.numel() == 0:
        return y_pred.new_tensor(0.0)
    diff = pos_scores.unsqueeze(1) - neg_scores.unsqueeze(0)
    return -F.logsigmoid(diff).mean()



def compute_joint_loss(logits_2d, gate_lambdas, y_true):
    calib_loss = F.cross_entropy(logits_2d, y_true.long())
    ranking_scores = logits_2d[:, 1] - logits_2d[:, 0]
    rank_loss = pairwise_bpr_loss(y_true, ranking_scores)
    avg_lambda = gate_lambdas.mean()
    total_loss = avg_lambda * rank_loss + (1.0 - avg_lambda) * calib_loss
    return total_loss, rank_loss, calib_loss, avg_lambda



def raw_predict_jrc(model, x_data, batch_size=256):
    loader = make_loader(x_data, y=None, batch_size=batch_size, shuffle=False)
    model.eval()
    logits_list = []
    gate_list = []
    with torch.no_grad():
        for batch in loader:
            batch = move_batch_to_device(batch)
            logits_2d, gate_lambdas = model(batch)
            logits_list.append(logits_2d.cpu().numpy())
            gate_list.append(gate_lambdas.cpu().numpy())
    logits_2d = np.concatenate(logits_list, axis=0)
    gate_lambdas = np.concatenate(gate_list, axis=0)
    return logits_2d, gate_lambdas



def softmax_second_col(logits_2d):
    shifted = logits_2d - logits_2d.max(axis=1, keepdims=True)
    exp_logits = np.exp(shifted)
    probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
    return probs[:, 1]



def train_jrc_model(
    model, x_train, y_train, x_val=None, y_val=None,
    epochs=5, batch_size=256, lr=0.001,
):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_loader = make_loader(x_train, y_train, batch_size=batch_size, shuffle=True)

    history = {'total_loss': [], 'rank_loss': [], 'calib_loss': [], 'gate_lambda': []}

    for epoch in range(epochs):
        epoch_metrics = {k: [] for k in history}
        t0 = time.time()
        model.train()

        for x_batch, y_batch in train_loader:
            x_batch = move_batch_to_device(x_batch)
            y_batch = y_batch.to(DEVICE)

            optimizer.zero_grad()
            logits_2d, gate_lambdas = model(x_batch)
            total_loss, r_loss, c_loss, avg_lam = compute_joint_loss(
                logits_2d, gate_lambdas, y_batch
            )
            total_loss.backward()
            optimizer.step()

            epoch_metrics['total_loss'].append(total_loss.item())
            epoch_metrics['rank_loss'].append(r_loss.item())
            epoch_metrics['calib_loss'].append(c_loss.item())
            epoch_metrics['gate_lambda'].append(avg_lam.item())

        elapsed = time.time() - t0
        for key in history:
            history[key].append(float(np.mean(epoch_metrics[key])))

        msg = (
            f"Epoch {epoch + 1}/{epochs} ({elapsed:.1f}s)  "
            f"Total: {history['total_loss'][-1]:.4f}  "
            f"Rank: {history['rank_loss'][-1]:.4f}  "
            f"Calib: {history['calib_loss'][-1]:.4f}  "
            f"Gate λ: {history['gate_lambda'][-1]:.4f}"
        )

        if x_val is not None and y_val is not None:
            val_logits, _ = raw_predict_jrc(model, x_val, batch_size=batch_size)
            val_calib = softmax_second_col(val_logits)
            val_auc = roc_auc_score(y_val, val_calib)
            msg += f"  | Val AUC: {val_auc:.4f}"

        print(msg)

    return model, history


print("✅ 训练函数定义完成")


## 6. 训练 JRC 模型

In [ ]:
print("🏋️ 开始训练 JRC 模型...")
print(f"   训练集样本数: {len(y_trn)}")
if offline:
    print(f"   验证集样本数: {len(y_val)}")
print()

model, train_history = train_jrc_model(
    model, x_trn, y_trn,
    x_val=x_val if offline else None,
    y_val=y_val if offline else None,
    epochs=5,
    batch_size=256,
    lr=0.001,
)

print("\n✅ JRC 模型训练完成！")

### JRC 模型训练日志指标说明
JRC（Joint Ranking & Calibration）模型核心是联合优化排序和校准任务，通过门控网络动态调整两者的损失权重。训练日志中各核心指标含义如下：

1. **Total（总损失）**
   总损失 $L$ 由排序损失和校准损失加权求和得到，计算公式为：
   $$L = \lambda \cdot L_{rank} + (1-\lambda) \cdot L_{calib}$$
   其中 $\lambda$ 为门控权重（Gate λ），用于动态调整两种损失的占比。

2. **Rank（排序损失）**
   即 $L_{rank}$，用于优化推荐排序效果（常用 BPR、Pairwise loss 等），核心目标是提升模型区分「点击样本」与「未点击样本」的能力。

3. **Calib（校准损失）**
   即 $L_{calib}$，通常采用交叉熵损失，用于优化模型输出概率的准确性，使预测概率尽可能接近真实点击概率。

4. **Gate λ（门控权重）**
   由独立的 DNN 网络输出（激活函数为 sigmoid），取值范围为 $[0,1]$，由模型自动学习。该权重决定当前 batch/样本中排序损失和校准损失的权重分配比例。

5. **Val AUC（验证集 AUC）**
   指验证集上的曲线下面积（Area Under Curve），是衡量模型排序能力的核心指标。AUC 值越高，代表模型区分正负样本的能力越强。



## 7. 预测与评估

JRC 模型输出 2D logit $\mathbf{z} = [z_0, z_1]$，从中可以衍生不同的预测分数：

| 模式 | 公式 | 适用场景 |
|------|------|----------|
| `calibration` | $p = \sigma(z_1 - z_0)$ | 需要准确 CTR 概率估计 |
| `ranking` | $s = z_1 - z_0$ (归一化) | 只关心排序质量 |
| `fusion` | $\lambda \cdot s_{norm} + (1-\lambda) \cdot p$ | 综合最优（推荐使用） |

In [ ]:
def predict_jrc(model, x_data, batch_size=256, mode='fusion'):
    """
    JRC 模型预测函数（PyTorch 2D Logit 版本）
    """
    logits_2d, gate_lambdas = raw_predict_jrc(model, x_data, batch_size=batch_size)

    # 排序分数 = z₁ - z₀ (log-odds，无界)
    ranking_scores = logits_2d[:, 1] - logits_2d[:, 0]

    # 校准概率 = softmax(z)[1] = σ(z₁ - z₀)
    calib_probs = softmax_second_col(logits_2d)

    if mode == 'calibration':
        return calib_probs
    elif mode == 'ranking':
        r_min, r_max = ranking_scores.min(), ranking_scores.max()
        if r_max > r_min:
            return (ranking_scores - r_min) / (r_max - r_min)
        return ranking_scores
    elif mode == 'fusion':
        r_min, r_max = ranking_scores.min(), ranking_scores.max()
        if r_max > r_min:
            norm_rank = (ranking_scores - r_min) / (r_max - r_min)
        else:
            norm_rank = ranking_scores
        return gate_lambdas * norm_rank + (1 - gate_lambdas) * calib_probs
    else:
        raise ValueError(f"Unknown mode: {mode}")


print("✅ 预测函数定义完成")


In [ ]:
# ====== 评估辅助函数 ======

def norm_sim(sim_df, weight=0.0):
    """分数归一化到 [0, 1]"""
    min_sim = sim_df.min()
    max_sim = sim_df.max()
    if max_sim == min_sim:
        sim_df = sim_df.apply(lambda sim: 1.0)
    else:
        sim_df = sim_df.apply(lambda sim: 1.0 * (sim - min_sim) / (max_sim - min_sim))
    sim_df = sim_df.apply(lambda sim: sim + weight)
    return sim_df


def calculate_mrr_metrics(df, topk=5, score_col='pred_score'):
    """计算 MRR@K 和 HR@K"""
    df_sorted = df.sort_values(by=['user_id', score_col], ascending=[True, False])
    mrr_sum = hr_sum = user_count = pos_user_count = 0

    for user_id, group in df_sorted.groupby('user_id'):
        user_count += 1
        labels = group['label'].values
        if labels.sum() == 0:
            continue
        pos_user_count += 1
        pos_idx = np.where(labels == 1)[0]
        if len(pos_idx) > 0:
            rank = pos_idx[0] + 1
            if rank <= topk:
                mrr_sum += 1.0 / rank
                hr_sum += 1

    mrr = mrr_sum / user_count if user_count > 0 else 0
    hr = hr_sum / user_count if user_count > 0 else 0
    print(f"   MRR@{topk}: {mrr:.5f}")
    print(f"   HR@{topk}:  {hr:.5f}")
    print(f"   有正样本用户: {pos_user_count}/{user_count}")
    return mrr, hr


def cal_ndcg(labels, preds, user_id_list, k=5):
    """分用户计算 NDCG@k"""
    group_score = defaultdict(list)
    group_truth = defaultdict(list)
    for idx, uid in enumerate(user_id_list):
        group_score[uid].append(preds[idx])
        group_truth[uid].append(labels[idx])

    ndcg_list = []
    for uid in group_score:
        y_true = np.array([group_truth[uid]])
        y_score = np.array([group_score[uid]])
        if np.sum(y_true) == 0:
            ndcg_list.append(0.0)
        else:
            ndcg_list.append(ndcg_score(y_true, y_score, k=k))
    return np.mean(ndcg_list)


def submit(recall_df, topk=5, model_name=None):
    """生成提交文件（仅在线模式使用）"""
    recall_df = recall_df.sort_values(by=['user_id', 'pred_score'])
    recall_df['rank'] = recall_df.groupby(['user_id'])['pred_score'].rank(
        ascending=False, method='first'
    )
    tmp = recall_df.groupby('user_id').apply(lambda x: x['rank'].max())
    assert tmp.min() >= topk, f"部分用户候选文章数不足 {topk} 篇！"

    del recall_df['pred_score']
    sub = recall_df[recall_df['rank'] <= topk].set_index(['user_id', 'rank']).unstack(-1).reset_index()
    sub.columns = [int(col) if isinstance(col, int) else col for col in sub.columns.droplevel(0)]
    sub = sub.rename(columns={
        '': 'user_id', 1: 'article_1', 2: 'article_2',
        3: 'article_3', 4: 'article_4', 5: 'article_5',
    })
    save_name = output_path / (model_name + '_' + datetime.today().strftime('%m-%d') + '.csv')
    sub.to_csv(save_name, index=False, header=True)
    print(f"📁 提交文件已保存: {save_name}")


print("✅ 评估辅助函数定义完成")

In [ ]:
# ====== 全面评估 JRC 模型（离线模式）======
if offline:
    print("📈 " + "=" * 60)
    print("📈  JRC 模型 - 验证集效果评估")
    print("📈 " + "=" * 60)

    # 获取两个输出: logits_2d [N,2] + gate_lambda [N,]
    val_logits_2d, val_gate_raw = raw_predict_jrc(model, x_val, batch_size=256)

    # 从 2D logit 衍生排序分数和校准概率
    val_ranking_scores = val_logits_2d[:, 1] - val_logits_2d[:, 0]
    val_calib_probs = softmax_second_col(val_logits_2d)

    print(f"\n🔍 2D Logit 统计:")
    print(f"   z₀ 均值: {val_logits_2d[:, 0].mean():.4f}, z₁ 均值: {val_logits_2d[:, 1].mean():.4f}")
    print(f"   排序分数 s=z₁-z₀ 范围: [{val_ranking_scores.min():.4f}, {val_ranking_scores.max():.4f}]")
    print(f"   校准概率 p=σ(s) 均值: {val_calib_probs.mean():.4f}")

    print(f"\n🔍 门控网络 (Context Gate) 统计:")
    print(f"   λ 均值: {val_gate_raw.mean():.4f}")
    print(f"   λ 标准差: {val_gate_raw.std():.4f}")
    print(f"   λ 范围: [{val_gate_raw.min():.4f}, {val_gate_raw.max():.4f}]")
    print(f"   → λ > 0.5 (偏排序): {(val_gate_raw > 0.5).mean():.2%}")
    print(f"   → λ < 0.5 (偏校准): {(val_gate_raw <= 0.5).mean():.2%}")

    modes = ['calibration', 'ranking', 'fusion']
    mode_labels = ['Calibration (校准概率 σ(z₁-z₀))', 'Ranking (排序分数 z₁-z₀)', 'Fusion (门控融合)']

    print(f"\n{'=' * 60}")
    for mode, label in zip(modes, mode_labels):
        preds = predict_jrc(model, x_val, mode=mode)
        auc = roc_auc_score(y_val, preds)
        ndcg = cal_ndcg(y_val, preds, val_df['user_id'].values, k=5)

        print(f"\n📊 【{label}】")
        print(f"   AUC:    {auc:.4f}")
        print(f"   NDCG@5: {ndcg:.4f}")

        eval_df = val_df[['user_id', 'click_article_id', 'label']].copy()
        eval_df['pred_score'] = preds
        calculate_mrr_metrics(eval_df, topk=5)

    print(f"\n{'=' * 60}")
    print("✅ 评估完成！")
else:
    print("⏭️ 在线模式跳过验证集评估")




#### 1. 2D Logit 统计
| 指标 | 含义 |
|------|------|
| z₀ 均值 / z₁ 均值 | 模型输出的两类 logit（未点击/点击）平均值|
| 排序分数 s=z₁-z₀ 范围 | 排序分数的最小值/最大值，衡量模型区分「点击」与「未点击」样本的能力边界 |
| 校准概率 p=σ(s) 均值 | 通过 softmax 转换得到的点击概率平均值，反映模型输出概率的整体分布特征 |

#### 2. 门控网络 (Context Gate) 统计
| 指标 | 含义 |
|------|------|
| λ 均值 / 标准差 / 范围 | 门控权重 λ 的统计特征，反映验证集上模型对「排序损失」和「校准损失」的权重分配规律 |
| λ > 0.5 (偏排序) | λ 大于 0.5 的样本占比，代表这些样本训练时更侧重优化排序损失 |
| λ < 0.5 (偏校准) | λ 小于等于 0.5 的样本占比，代表这些样本训练时更侧重优化校准损失 |

#### 3. 三种预测方式效果对比
模型支持三种预测策略，每种策略均输出 AUC、NDCG@5、MRR 核心指标，具体说明：

| 预测方式 | 计算逻辑 | 适用场景 | 评估指标含义 |
|----------|----------|----------|--------------|
| Calibration | 基于校准概率（softmax 输出）预测 | 概率预估类任务（如点击率预测） |  AUC：排序能力<br>-NDCG@5：前5个推荐结果的排序质量<br> MRR：推荐列表中首个正确结果的位置 |
| Ranking | 基于排序分数（z₁-z₀）预测 | 纯排序类任务（如推荐列表排序） | 同上 |
| Fusion | 门控融合（结合排序分数+校准概率） | 需要兼顾排序效果和概率准确性的场景 | 同上 |

## 8. 五折交叉验证 + 生成 OOF 特征

使用五折交叉验证生成 out-of-fold 预测，用于后续 Stacking 融合。

In [ ]:
print("🔄 开始 JRC 模型五折交叉验证...")


def get_kfold_users(df, n=5):
    user_ids = df['user_id'].unique()
    return [user_ids[i::n] for i in range(n)]


k_fold = 5
user_set = get_kfold_users(trn_df, n=k_fold)

score_list = []
score_df = trn_df[['user_id', 'click_article_id', 'label']].copy()

# 初始化累加数组（根据模式和测试集是否为空决定）
if not offline and len(tst_df) > 0:
    sub_preds = np.zeros(len(tst_df))
elif offline:
    val_sub_preds = np.zeros(len(val_df))

for n_fold, valid_user in enumerate(user_set):
    print(f"\n   📁 Fold {n_fold + 1}/{k_fold}")
    fold_trn = trn_df[~trn_df['user_id'].isin(valid_user)]
    fold_val = trn_df[trn_df['user_id'].isin(valid_user)]

    x_fold_trn = build_model_input(fold_trn)
    y_fold_trn = fold_trn['label'].values.astype('float32')
    x_fold_val = build_model_input(fold_val)
    y_fold_val = fold_val['label'].values.astype('float32')

    fold_model = init_jrc_model()
    fold_model, _ = train_jrc_model(
        fold_model, x_fold_trn, y_fold_trn,
        x_val=x_fold_val, y_val=y_fold_val,
        epochs=3, batch_size=256, lr=0.001,
    )

    fold_val_preds = predict_jrc(fold_model, x_fold_val, mode='fusion')
    fold_val_copy = fold_val[['user_id', 'click_article_id']].copy()
    fold_val_copy['pred_score'] = fold_val_preds
    fold_val_copy['pred_score'] = fold_val_copy['pred_score'].transform(lambda x: norm_sim(x))
    fold_val_copy['pred_rank'] = fold_val_copy.groupby('user_id')['pred_score'].rank(
        ascending=False, method='first'
    )
    score_list.append(fold_val_copy)

    if not offline and x_tst is not None:
        sub_preds += predict_jrc(fold_model, x_tst, mode='fusion')
    elif offline:
        val_sub_preds += predict_jrc(fold_model, x_val, mode='fusion')

    del fold_model
    gc.collect()
    if TORCH_AVAILABLE and torch.cuda.is_available():
        torch.cuda.empty_cache()

score_df_ = pd.concat(score_list, axis=0)
score_df = score_df.merge(score_df_, how='left', on=['user_id', 'click_article_id'])
score_df[['user_id', 'click_article_id', 'pred_score', 'pred_rank', 'label']].to_csv(
    save_path / 'trn_jrc_feats_all.csv', index=False
)
print(f"\n   ✅ 训练集 OOF 特征已保存: trn_jrc_feats_all.csv")

if not offline and x_tst is not None:
    tst_df['pred_score'] = sub_preds / k_fold
    tst_df['pred_score'] = tst_df['pred_score'].transform(lambda x: norm_sim(x))
    tst_df['pred_rank'] = tst_df.groupby('user_id')['pred_score'].rank(
        ascending=False, method='first'
    )
    tst_df[['user_id', 'click_article_id', 'pred_score', 'pred_rank']].to_csv(
        save_path / 'tst_jrc_feats_all.csv', index=False
    )
    print(f"   ✅ 测试集 CV 特征已保存: tst_jrc_feats_all.csv")

    rank_results = tst_df[['user_id', 'click_article_id', 'pred_score']].copy()
    rank_results['click_article_id'] = rank_results['click_article_id'].astype(int)
    submit(rank_results, topk=5, model_name='jrc_cv_all')
elif not offline and x_tst is None:
    print("   ⚠️ 测试集为空，跳过 CV 特征保存和提交文件生成")
    print("   请检查前面的 4.recall_all.ipynb 和 5.feature_engineering_all.ipynb")
else:
    val_df_jrc = val_df[['user_id', 'click_article_id', 'label']].copy()
    val_df_jrc['pred_score'] = val_sub_preds / k_fold
    val_df_jrc['pred_score'] = val_df_jrc['pred_score'].transform(lambda x: norm_sim(x))
    val_df_jrc.to_csv(save_path / 'jrc_val_score_all.csv', index=False)
    print(f"   ✅ 验证集预测已保存: jrc_val_score_all.csv")

    print(f"\n📊 【JRC 五折 CV - 验证集详细指标】")
    cv_auc = roc_auc_score(val_df_jrc['label'], val_df_jrc['pred_score'])
    cv_ndcg = cal_ndcg(
        val_df_jrc['label'].values, val_df_jrc['pred_score'].values,
        val_df_jrc['user_id'].values, k=5,
    )
    print(f"   AUC:    {cv_auc:.4f}")
    print(f"   NDCG@5: {cv_ndcg:.4f}")
    calculate_mrr_metrics(val_df_jrc, topk=5)

print("\n✅ JRC 五折交叉验证完成！")


## 9. 在线模式 - 生成提交文件（单模型）

如果 `MODE = "online"`，使用全量训练好的 JRC 模型预测测试集并生成提交文件。

In [ ]:
if not offline:
    if x_tst is not None and len(tst_df) > 0:
        print("🔮 [在线模式] 使用全量模型预测测试集...")

        for mode in ['calibration', 'ranking', 'fusion']:
            preds = predict_jrc(model, x_tst, mode=mode)
            result_df = tst_df[['user_id', 'click_article_id']].copy()
            result_df['pred_score'] = preds
            result_df['click_article_id'] = result_df['click_article_id'].astype(int)

            result_df.to_csv(output_path / f'jrc_{mode}_score_all.csv', index=False)
            submit(result_df.copy(), topk=5, model_name=f'jrc_{mode}_all')

        print("✅ 所有提交文件生成完成！")
    else:
        print("⚠️ [在线模式] 测试集为空，无法生成提交文件")
        print("   请检查前面的 4.recall_all.ipynb 和 5.feature_engineering_all.ipynb")
else:
    print("⏭️ [离线模式] 跳过提交文件生成")


## 10. 总结

### 本 Notebook 实现了什么？

我们将 **"Joint Optimization of Ranking and Calibration with Contextualized Hybrid Model"**
论文的核心思想应用于新闻推荐排序任务：

1. **Shared Backbone**：基于 DIN 的共享特征提取器，捕捉用户兴趣与候选文章的关联
2. **2D Logit Head**（核心创新）：输出 $\mathbf{z} = [z_0, z_1]$，从同一表示解耦出：
   - 排序分数 $s = z_1 - z_0$（无界 log-odds，梯度不饱和）
   - 校准概率 $p = \sigma(s)$（proper scoring rule，保证校准）
3. **Context Gate**：动态学习每个样本的排序/校准权重 $\lambda$
4. **Joint Loss**：$L = \lambda \cdot L_{rank}(s) + (1 - \lambda) \cdot L_{calib}(\mathbf{z})$

### 2D Logit 的核心优势

| 传统 1D | JRC 2D |
|---------|--------|
| $p = \sigma(z)$，排序和校准用同一个标量 | $\mathbf{z} = [z_0, z_1]$，排序 $s = z_1 - z_0$，校准 $p = \sigma(s)$ |
| sigmoid 饱和导致排序梯度消失 | $s$ 无界，梯度不饱和 |
| BCE 同时承担排序 + 校准 | BPR on $s$ + CE on $\mathbf{z}$，各自保持最优性质 |

### 与原始 `6.ranking.ipynb` 的对比

| 维度 | 原始方案 | JRC 方案 |
|------|---------|----------|
| 模型数量 | LGBMRanker + LGBMClassifier + DIN = 3 个 | JRC = **1 个** |
| 输出设计 | 各模型独立 1D 输出 | 2D Logit 统一表示 |
| 融合方式 | 手动权重 0.3/0.4/0.3 | Context Gate **自动学习** |
| 上下文自适应 | ❌ 静态权重 | ✅ 每样本动态 $\lambda$ |

### 关键超参数调优建议

| 超参数 | 当前值 | 调优方向 |
|--------|--------|----------|
| `shared_dnn_units` | [256, 128] | 增大可提升表达力（如 [512, 256]），但可能过拟合 |
| `logit_head_units` | [64] | 2D Logit Head 隐层，可尝试 [128] 或 [64, 32] |
| `epochs` | 5 | 监控 Val AUC 选择最佳 epoch |
| `batch_size` | 256 | 增大 batch 有利于 BPR Loss（更多正负对） |
| `lr` | 0.001 | 尝试 warmup + cosine decay |
| `dropout_rate` | 0.1 | 防过拟合，0.1~0.3 |
| `emb_dim` | 8 | 可以尝试 16 或 32 |